# Adding MC systematics (GENIE, Flux, G4, MCstat)

Covers adding rate/norm/MCstat systematics on top of step2a's MC+data
combination, via `funcs.py`'s `systs_input()`. Detector-variation
systematics (`'detv'`) are added separately in
`step4_add_detvar_systematics.ipynb`.

| Section | File | What you set |
|---|---|---|
| 1 | -- | produce `MC_FILE`/`DATA_FILE`/`OFFBEAM_FILE` via `run_df_maker.py` |
| 2 | -- | load MC + on-beam + off-beam (cuts applied) + off-beam scaling |
| 3 | `funcs.py` | add systematics with `systs_input()` |
| 4 | `plotting.py` | plot the systematics category breakdown |

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# TODO: set to the PARENT directory of your cafpybara/ clone -- NOT cafpybara/ itself (skipped if already importable).
sys.path.append("TODO_SET_ME/parent_dir_of_cafpybara")

# TODO: set to your cafpyana clone (skipped if already exported).
os.environ.setdefault("CAFPYANA_PATH", "TODO_SET_ME/path/to/cafpyana")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cafpybara import core

# TODO: import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

## 1. Make `MC_FILE`/`DATA_FILE`/`OFFBEAM_FILE` with `cafpyana`

`MC_FILE` needs real event weights (GENIE/Flux/Geant4) too -- typically
produced as multiple grid-job shards (`-ngrid N`) merged into one file:

1. `cd $CAFPYANA_PATH`
2. ```bash
   python run_df_maker.py -c ./analysis_village/hnl_nuee_nupi0/configs/hnl_mcnu_wgt.py -l <input.list> -o <output_name> -ngrid <N>
   ```
3. Merge the `N` per-shard outputs into one file:
   ```bash
   python ./analysis_village/hnl_nuee_nupi0/scripts/concat_hdf.py <shard_output_dir> -o <merged_output_dir>
   ```

`-ngrid N` round-robins the input list across `N` grid jobs, each
writing its own shard; `concat_hdf.py` merges them and stamps each
row's source shard as a `file_idx` level (needed for `mcstat`, section 2).

`DATA_FILE`/`OFFBEAM_FILE` use the same config/grid/merge combo:

```bash
python run_df_maker.py -c ./analysis_village/hnl_nuee_nupi0/configs/hnl_data.py -l <input.list> -o <output_name> -ngrid <N>
python ./analysis_village/hnl_nuee_nupi0/scripts/concat_hdf.py <shard_output_dir> -o <merged_output_dir>
```

Both configs leave `make_hnldf`'s `applyPreselection=True`/
`savePfp=False` defaults untouched: a slice-level cosmic/nu_score/FV
cut, then a 0-track + 1-or-2-shower topology cut that flattens the
surviving showers into `primshw`/`secshw` column groups (not a generic
`pfp` table). `DEFAULT_CUTS` is a separate, topology-specific cut
layered on top.

> ⚠️ that's why `VAR` below is `('primshw', 'shw', 'bestplane_energy', '', '', '')`, not step1a/step2a's `('pfp', 'shw', ...)`.

* **TODO:** set `MC_FILE`, `DATA_FILE`, `OFFBEAM_FILE`

In [ ]:
#MC_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/nu2026_df/mcbnb_cv/mcbnb_wgt_full_v4.df"
MC_FILE = "/path/to/your_mc_file.h5"
#DATA_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/nu2026_df/dtbnb/fixed_bnblight_2026a_v4.df"
DATA_FILE = "/path/to/your_data_file.h5"
#OFFBEAM_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/nu2026_df/dtoff/offbeamlight_v4.df"
OFFBEAM_FILE = "/path/to/your_offbeam_file.h5"
#VAR = "your_variable"
#BINS = [0, 1, 2, 3]
VAR = ('primshw', 'shw', 'bestplane_energy', '', '', '') #TEST
BINS = np.linspace(0, 1, 10) #TEST

## 2. Load MC + data with cuts applied, then off-beam scaling

Same single load step as step2a's A4, reused here for the systematics demo.

* **TODO:**
  * pass `preprocess_fn=getattr(ana, 'preprocess_databnb', ana.preprocess_data)`
    to the on-beam call, and the off-beam equivalent to the off-beam call --
    see step2a A3/A4; falls back to the no-op `preprocess_data` if unset.
  * double-check `f` is the most up-to-date off-beam scaling fudge factor
    (see step2a A4).

In [ ]:
try:
    mc_df, mc_pot, mc_ngen = ana.load_mc(MC_FILE, cuts=ana.DEFAULT_CUTS)

    data_df, data_pot, data_ngates = ana.load_data(
        DATA_FILE, cuts=ana.DEFAULT_CUTS,
        preprocess_fn=getattr(ana, 'preprocess_databnb', ana.preprocess_data),
    )

    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False,
        offbeam_signal_value=ana.signal_dict["offbeam"],
        cuts=ana.DEFAULT_CUTS,
        preprocess_fn=getattr(ana, 'preprocess_dataoff', ana.preprocess_data),
    )

    f = 0.0725  # TODO: double check for the most-up-to-date fudge factor
    ongates_per_pot = data_ngates / data_pot
    scale_offbeam = ((1 - f) * (ongates_per_pot * mc_pot)) / offbeam_ngates

    weight_col = ('weights_mc', '', '', '', '', '')
    mc_df[weight_col] = 1.0
    offbeam_df[weight_col] = scale_offbeam

    mcstat_cols = ['__ntuple', 'entry', 'rec.slc..index', 'run', 'subrun', 'evt', 'sample']
    mc_top_level = set(mc_df.columns.get_level_values(0)) | set(mc_df.index.names)
    if 'file_idx' in mc_top_level:
        mcstat_cols.append('file_idx')  # only from concat_hdf.py (see section 1 above) -- fine if absent here

    mc_df = pd.concat([
        ana.mcstat(mc_df.assign(sample=0).set_index("sample", append=True).reset_index(), cols=mcstat_cols),
        offbeam_df.assign(sample=1).set_index("sample", append=True).reset_index(),
    ])
    print("Merged MC + off-beam rows:", len(mc_df))

    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot)  # no systs= yet
except Exception as e:
    mc_df, data_df = None, None
    print("No real MC/data/offbeam file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## 3. Add systematics with `funcs.py`'s `systs_input()`

`systs_input()` builds a `SystematicsInput` from `mcbnb_pot` +
`uncertainty_keys` (`'rate'`, `'xsec'`, `'detv'`, `'norm'`, `'cosmic'`),
reusing `DEFAULT_CUTS`/`define_signal` from `analysis.py`. Pass it to
`plot_var(..., systs=si)`/`plot_mc_data(..., systs=si)`, or unpack
`si.to_kwargs()` into `get_total_cov`.

* **TODO:** none new -- reuses `DEFAULT_CUTS`/`define_signal` from `analysis.py`.

> ⚠️ `'cosmic'` and `'detv'` aren't included here -- `'detv'` is added in `step4_add_detvar_systematics.ipynb`; `'cosmic'` needs an in-time-cosmic sample, not required by default.

In [ ]:
print("ana.get_total_cov IS core.funcs.get_total_cov:", ana.get_total_cov is core.funcs.get_total_cov)
print("(if this were ever False for a new analysis, get_total_cov has been wrongly overridden)\n")

if mc_df is not None:
    si = ana.systs_input(mc_pot, uncertainty_keys={'rate', 'norm'})
    mc_syst_output = ana.get_total_cov(mc_df, reco_var=VAR, bins=BINS, **si.to_kwargs())
    print("mc_syst_output.rate_cov shape:", mc_syst_output.rate_cov.shape)
    print("mc_syst_output.rate_syst_dict keys:", list(mc_syst_output.rate_syst_dict.keys()))
else:
    mc_syst_output = None
    print("No real MC file here (this notebook needs none) -- see section 2 above")

In [ ]:
# plot_mc_data with systematics on (reuses mc_syst_output from above):
if mc_df is not None and data_df is not None:
    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot, systs=mc_syst_output)
else:
    print("No real data file here (this notebook needs none) -- see section 2 above")

## 4. Plot the systematics category breakdown

* **TODO (optional):** inspect `plot_syst_category_breakdown`'s docstring
  for plotting options (figure size, projected POT, xsec mode, etc.)

In [ ]:
if mc_syst_output is not None:
    fig, axes, cats, cat_sums = ana.plot_syst_category_breakdown(
        syst_vars=[(mc_syst_output, BINS, VAR)],
        category_dict=ana.category_dict_signal,
    )
    plt.show()
else:
    print("No real MC file here (this notebook needs none) -- see section 3 above")